# Gradient Sports API — Exemplos de Uso

Importa o cliente de `gradient_client.py`.  
A autenticação é lida automaticamente do arquivo `.env` (`BEARER_TOKEN`).

In [1]:
import pandas as pd
import sys
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(str(Path().resolve().parent.parent))
from src.gradient_client import GradientSportsClient

pd.set_option('display.max_columns', None)

### Status da requisição

In [2]:
client = GradientSportsClient()

# health check
client.get_status()

{'data': {'status': 'ok'}}

### Competições

In [3]:
# competitions & seasons the account can access
df_competitions = client.get_competitions(as_dataframe=True).drop('datasets', axis=1).drop_duplicates().reset_index(drop=True)
df_competitions

,season,competition.id,competition.name
0,2020-2021,1,Premier League
1,2021-2022,1,Premier League
2,2022-2023,1,Premier League
3,2023-2024,1,Premier League
4,2024-2025,1,Premier League
5,2025-2026,1,Premier League
6,2023,42,Brasileiro Série A
7,2024,42,Brasileiro Série A
8,2025,42,Brasileiro Série A
9,2026,42,Brasileiro Série A


- Existem dados de duas competições, Premier League e Brasileiro Série A.
- Dentre essas competições, existem dados de 6 temporadas da Premier League e 4 temporadas do Brasileiro Série A.

In [4]:
# teams the account can access
df_teams = client.get_teams(as_dataframe=True).drop('dataset', axis=1).drop_duplicates().reset_index(drop=True)
df_teams.sort_values('competition.id')

,team.id,team.name,competition.id,competition.name
0,1.0,AFC Bournemouth,1,Premier League
1,20.0,Wolverhampton Wanderers,1,Premier League
6,221.0,Nottingham Forest,1,Premier League
7,335.0,Sunderland AFC,1,Premier League
12,10.0,Liverpool,1,Premier League
13,218.0,Luton Town,1,Premier League
11,7.0,Crystal Palace,1,Premier League
8,16.0,Southampton,1,Premier League
24,55.0,Leeds United,1,Premier League
25,13.0,Newcastle United,1,Premier League


In [5]:
df_teams.info()

<class 'pandas.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   team.id           57 non-null     float64
 1   team.name         57 non-null     str    
 2   competition.id    59 non-null     int64  
 3   competition.name  59 non-null     str    
dtypes: float64(1), int64(1), str(2)
memory usage: 2.0 KB


In [6]:
df_teams.duplicated().sum()

np.int64(0)

- O dataset de times possui duas colunas com ids dos times nulas e não possui nenhuma duplicata. As colunas com ids dos times nulas devem ser removidas.

In [7]:
df_teams = df_teams[df_teams['team.name'].notna()].reset_index()

In [8]:
df_teams['competition.name'].value_counts()

competition.name
Brasileiro Série A    29
Premier League        28
Name: count, dtype: int64

- Entre as temporadas, existem dados de 28 times da Premier League e 29 do Brasileirão.

### Jogos

In [9]:
# all games — or filter by season / competition / team
# as_dataframe=True → one row per game, nested dicts dot-expanded
games = client.get_games(as_dataframe=True)
games

,id,date,season,teamExtraTimeStartSide,teamStartSide,venueType,team.id,team.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name,stadium.name,stadium.length,stadium.width
0,32019,2024-08-31,2024-2025,Left,Right,OPPONENT_HOME,3,Aston Villa,1,Premier League,9,Leicester City,King Power Stadium,105.0,68.0
1,4646,2023-02-04,2022-2023,Right,Right,TEAM_HOME,3,Aston Villa,1,Premier League,9,Leicester City,Villa Park,105.0,68.0
2,259,2020-12-07,2020-2021,Right,Left,TEAM_HOME,4,Brighton & Hove Albion,1,Premier League,16,Southampton,American Express Stadium,105.0,68.0
3,40871,2025-08-17,2025-2026,Left,Right,OPPONENT_HOME,119,Brentford,1,Premier League,221,Nottingham Forest,The City Ground,102.5,68.0
4,32192,2025-01-04,2024-2025,Left,Right,OPPONENT_HOME,119,Brentford,1,Premier League,16,Southampton,St. Mary's Stadium,105.0,68.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3407,13556,2024-02-04,2023-2024,Right,Left,TEAM_HOME,1,AFC Bournemouth,1,Premier League,221,Nottingham Forest,Vitality Stadium,105.0,68.0
3408,4725,2023-04-08,2022-2023,Right,Right,TEAM_HOME,3,Aston Villa,1,Premier League,221,Nottingham Forest,Villa Park,105.0,68.0
3409,186,2020-10-04,2020-2021,Right,Left,TEAM_HOME,12,Manchester United,1,Premier League,17,Tottenham Hotspur,Old Trafford,105.0,68.0
3410,37019,2025-11-15,2025,Left,Left,OPPONENT_HOME,435,Flamengo,42,Brasileiro Série A,873,Sport Recife,Estádio Ilha do Retiro,105.0,68.0


In [10]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 3412 entries, 0 to 3411
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      3412 non-null   int64  
 1   date                    3412 non-null   str    
 2   season                  3412 non-null   str    
 3   teamExtraTimeStartSide  3412 non-null   str    
 4   teamStartSide           3412 non-null   str    
 5   venueType               3412 non-null   str    
 6   team.id                 3412 non-null   int64  
 7   team.name               3412 non-null   str    
 8   competition.id          3412 non-null   int64  
 9   competition.name        3412 non-null   str    
 10  opponentTeam.id         3412 non-null   int64  
 11  opponentTeam.name       3412 non-null   str    
 12  stadium.name            3412 non-null   str    
 13  stadium.length          3412 non-null   float64
 14  stadium.width           3412 non-null   float64
dty

In [11]:
games.duplicated().sum()

np.int64(0)

- O conjunto de dados de jogos não possui linhas duplicadas ou dados ausentes.

In [12]:
games['competition.name'].value_counts()

competition.name
Premier League        2205
Brasileiro Série A    1207
Name: count, dtype: int64

- Existem 2205 partidas da Premier League e 1207 do Brasileirão.

In [ ]:
games.groupby(['competition.name', 'season'])['team.name'].nunique()

- No Brasileiro Série A, existem dados de 19 times nas temporadas de 2023 a 2025, enquanto que 2026 possui dados só de 17 times.
- Na Premier League existem dados de todos os times.

In [13]:
games[['competition.name', 'season']].value_counts().sort_index()

competition.name    season   
Brasileiro Série A  2023         377
                    2024         376
                    2025         379
                    2026          75
Premier League      2020-2021    378
                    2021-2022    377
                    2022-2023    380
                    2023-2024    381
                    2024-2025    380
                    2025-2026    309
Name: count, dtype: int64

- Nota-se que os dados não apresentam redundâncias (exemplo: Time A x Time B / Time B x Time A) e completam 380 partidas, no máximo.
- Na temporada 2023-2024 há um dado a mais, indicando uma incosistência nos dados.
- Existem temporadas faltando dados, exceto 2025-2026 (Premier League) e 2026 (Brasileirão) que ainda estão em andamento.

In [ ]:
# encontrar uma forma de gerar o gráfico de partidas considerar o team e opponent team

In [38]:
# =========================
# 1. FILTROS + AGREGAÇÃO
# =========================

df_pl = (
    games[games["competition.name"] == "Premier League"]
    .groupby(["season", "team.name"])
    .size()
    .reset_index(name="matches")
)

df_br = (
    games[games["competition.name"] == "Brasileiro Série A"]
    .groupby(["season", "team.name"])
    .size()
    .reset_index(name="matches")
)

pl_seasons = sorted(df_pl["season"].unique())
br_seasons = sorted(df_br["season"].unique())

# =========================
# 2. PLOT PREMIER LEAGUE
# =========================

fig_pl = go.Figure()

for season in pl_seasons:
    df_season = df_pl[df_pl["season"] == season]

    fig_pl.add_trace(
        go.Bar(
            x=df_season["team.name"],
            y=df_season["matches"],
            name=str(season),
        )
    )

fig_pl.update_layout(
    title="Premier League — jogos por time e temporada",
    height=500,
    width=1500,
    barmode="group"
)

fig_pl.update_xaxes(tickangle=45)

fig_pl.add_hline(
    y=38,
    line_dash="dash",
    line_color="red",
    annotation_text="máximo de partidas",
    annotation_position="bottom right"
)

fig_pl.show()

# =========================
# 3. PLOT BRASILEIRÃO
# =========================

fig_br = go.Figure()

for season in br_seasons:
    df_season = df_br[df_br["season"] == season]

    fig_br.add_trace(
        go.Bar(
            x=df_season["team.name"],
            y=df_season["matches"],
            name=str(season),
        )
    )

fig_br.update_layout(
    title="Brasileirão Série A — jogos por time e temporada",
    height=500,
    width=1500,
    barmode="group"
)

fig_br.update_xaxes(tickangle=45)

fig_br.add_hline(
    y=38,
    line_dash="dash",
    line_color="red",
    annotation_text="máximo de partidas",
    annotation_position="bottom right"
)

fig_br.show()

- Pode-se notar que, conforme as temporadas disponiveis, a distribuição de dados disponíveis dos times varia bastante.
- Em nenhuma temporada há dados completos dos 20 times disputando na temporada.
- Curiosamente, o time da Premier League 'AFC Bournemouth' na temporada 2023-2024 teve uma partida a mais, ou seja, uma inconsistência nos dados.

In [41]:
games.loc[(games['season'] == '2023-2024') & (games['team.name'] == 'AFC Bournemouth')].value_counts('opponentTeam.name')

opponentTeam.name
Luton Town                 3
Liverpool                  2
Wolverhampton Wanderers    2
Chelsea                    2
Newcastle United           2
West Ham                   2
Manchester City            2
Arsenal                    2
Crystal Palace             2
Aston Villa                2
Brighton & Hove Albion     2
Manchester United          2
Tottenham Hotspur          2
Sheffield United           2
Fulham                     2
Everton                    2
Nottingham Forest          2
Burnley                    2
Brentford                  2
Name: count, dtype: int64

In [42]:
games.loc[(games['season'] == '2023-2024') & 
          (games['team.name'] == 'AFC Bournemouth') &
          (games['opponentTeam.name'] == 'Luton Town')]

,id,date,season,teamExtraTimeStartSide,teamStartSide,venueType,team.id,team.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name,stadium.name,stadium.length,stadium.width
1944,13496,2023-12-16,2023-2024,Right,Right,TEAM_HOME,1,AFC Bournemouth,1,Premier League,218,Luton Town,Vitality Stadium,105.0,68.0
2011,20480,2024-03-13,2023-2024,Right,Right,TEAM_HOME,1,AFC Bournemouth,1,Premier League,218,Luton Town,Vitality Stadium,105.0,68.0
2981,13650,2024-04-06,2023-2024,Left,Left,OPPONENT_HOME,1,AFC Bournemouth,1,Premier League,218,Luton Town,Kenilworth Road,100.0,66.0


- Nota-se que um dos registros está incosistente pela data e gerou uma duplicata. A data '2023-12-16' está incorreta, já que esse jogo foi em '2024-03-13'.